# Spark Preparation
We check if we are in Google Colab.  If this is the case, install all necessary packages.

To run spark in Colab, we need to first install all the dependencies in Colab environment i.e. Apache Spark 3.3.2 with hadoop 3.3, Java 8 and Findspark to locate the spark in the system. The tools installation can be carried out inside the Jupyter Notebook of the Colab.
Learn more from [A Must-Read Guide on How to Work with PySpark on Google Colab for Data Scientists!](https://www.analyticsvidhya.com/blog/2020/11/a-must-read-guide-on-how-to-work-with-pyspark-on-google-colab-for-data-scientists/)

In [1]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

In [2]:
if IN_COLAB:
    !apt-get install openjdk-8-jdk-headless -qq > /dev/null
    !wget -q https://dlcdn.apache.org/spark/spark-3.3.2/spark-3.3.2-bin-hadoop3.tgz
    !tar xf spark-3.3.2-bin-hadoop3.tgz
    !mv spark-3.3.2-bin-hadoop3 spark
    !pip install -q findspark
    import os
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    os.environ["SPARK_HOME"] = "/content/spark"

# Start a Local Cluster

In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.master('local').appName("App1").getOrCreate()
spark

# Spark Assignment

Based on the movie review dataset in 'netflix-rotten-tomatoes-metacritic-imdb.csv', answer the below questions.

**Note:** do not clean or remove missing data

In [5]:
df = spark.read.csv("netflix-rotten-tomatoes-metacritic-imdb.csv", header=True, inferSchema=True)
df.printSchema()

root
 |-- Title: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Tags: string (nullable = true)
 |-- Languages: string (nullable = true)
 |-- Series or Movie: string (nullable = true)
 |-- Hidden Gem Score: double (nullable = true)
 |-- Country Availability: string (nullable = true)
 |-- Runtime: string (nullable = true)
 |-- Director: string (nullable = true)
 |-- Writer: string (nullable = true)
 |-- Actors: string (nullable = true)
 |-- View Rating: string (nullable = true)
 |-- IMDb Score: string (nullable = true)
 |-- Rotten Tomatoes Score: string (nullable = true)
 |-- Metacritic Score: string (nullable = true)
 |-- Awards Received: double (nullable = true)
 |-- Awards Nominated For: double (nullable = true)
 |-- Boxoffice: string (nullable = true)
 |-- Release Date: string (nullable = true)
 |-- Netflix Release Date: string (nullable = true)
 |-- Production House: string (nullable = true)
 |-- Netflix Link: string (nullable = true)
 |-- IMDb Link: string (null

In [8]:
cols = [col.replace(" ", "_")  for col in df.columns]
df = df.toDF(*cols)
df.printSchema()

root
 |-- Title: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Tags: string (nullable = true)
 |-- Languages: string (nullable = true)
 |-- Series_or_Movie: string (nullable = true)
 |-- Hidden_Gem_Score: double (nullable = true)
 |-- Country_Availability: string (nullable = true)
 |-- Runtime: string (nullable = true)
 |-- Director: string (nullable = true)
 |-- Writer: string (nullable = true)
 |-- Actors: string (nullable = true)
 |-- View_Rating: string (nullable = true)
 |-- IMDb_Score: string (nullable = true)
 |-- Rotten_Tomatoes_Score: string (nullable = true)
 |-- Metacritic_Score: string (nullable = true)
 |-- Awards_Received: double (nullable = true)
 |-- Awards_Nominated_For: double (nullable = true)
 |-- Boxoffice: string (nullable = true)
 |-- Release_Date: string (nullable = true)
 |-- Netflix_Release_Date: string (nullable = true)
 |-- Production_House: string (nullable = true)
 |-- Netflix_Link: string (nullable = true)
 |-- IMDb_Link: string (null

## What is the maximum and average of the overall hidden gem score?

In [12]:
from pyspark.sql.functions import max, avg
m, a = df.select(max(df["Hidden_Gem_Score"]), avg(df["Hidden_Gem_Score"])).collect()[0][:]
print(round(m,2), round(a,2))

9.8 5.94


## How many movies that are available in Korea?

In [15]:
print(df.filter(df["Languages"].contains("Korea")).count())
print(df.filter(df["Languages"].contains("Korea")).select(df["Title"]).distinct().count())
print(df.filter(df["Country_Availability"].contains("Korea")).count())
print(df.filter(df["Country_Availability"].contains("Korea")).select(df["Title"]).distinct().count())

735
724
4845
4789


## Which director has the highest average hidden gem score?

In [21]:
print(df.groupBy("Director").agg({"Hidden_Gem_Score":"avg"}).orderBy("avg(Hidden_Gem_Score)", ascending=False).collect()[0][:])
group_df = df.groupBy("Director").agg({"Hidden_Gem_Score":"avg"})
m = group_df.select(max(group_df["avg(Hidden_Gem_Score)"])).collect()[0][0]
print(group_df.filter(group_df["avg(Hidden_Gem_Score)"] == m).collect()[0][:])

('Dorin Marcu', 9.8)
('Dorin Marcu', 9.8)


## How many genres are there in the dataset?

In [40]:
from pyspark.sql import functions
print(df.filter(df["Genre"].isNull()).count())
print(df.filter(df["Genre"].isNotNull()).select("Genre").withColumn("Genres_split", functions.split(functions.col("Genre"), ", ")).withColumn("Genre_explode", functions.explode(functions.col("Genres_split"))).groupBy("Genre_explode").agg({"Genre":"count"}).count())

1710
28
